## Appendix 3: Sequence/Time Series Regression and Classification

Estimated time: 40-45 minutes
Accelerator : T4 GPU

This lab uses various neural network architectures - Fully connected, Convolutional and Recurrent to demonstrate their use for Time Series data regression and classification.  Constrained Single Board Computer (SBC) based edge devices are good with CNN based time series data classification.  More powerful inference servers support recurrent networks for time series regression.

The goal of this lab is to show you:
- To prepare time series data for supervised learning
- Use of Keras Timeseries Generator API
- RNN models - LSTM and GRU for time series regression
- CNN model for time series data classification

##### **Step 1:** Supervised learning for data in a sequence.

Run sample code and experiment with seq_length and batch size.

The code below uses Keras timeseries_dataset_from_array API to prepare sequence data for training.

In [0]:
from tensorflow import keras
from keras.utils import timeseries_dataset_from_array
from numpy import array
from keras.models import Sequential
from keras.layers import Dense

input_seq = array([0,1,1,2,3,5,8,13,21,34,55,89])

training_dataset = keras.utils.timeseries_dataset_from_array(
    data=input_seq[:],
    targets=input_seq[3:],
    sequence_length=3,
    batch_size=2,
)

for inputs, targets in training_dataset:
    for i in range(inputs.shape[0]):
        print([int(x) for x in inputs[i]], int(targets[i]))

##### **Step 2** - Create a model using fully connected network to capture the working of a Fibonacci series.


In [0]:
from tensorflow import keras
from keras.utils import timeseries_dataset_from_array
from numpy import array
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Input

input_seq = array([0,1,1,2,3,5,8,13,21,34,55,89])

seq_length = 3

training_dataset = keras.utils.timeseries_dataset_from_array(
    data=input_seq[:],
    targets=input_seq[3:],
    sequence_length=3,
    batch_size=1,
)

model = Sequential()
model.add(Input(shape=(seq_length,)))
model.add(Dense(100, activation='relu'))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mse')

model.fit(training_dataset, epochs=200, verbose=0)

x_input = array([34, 55,89]).reshape((1, seq_length))
y_output = model.predict(x_input, verbose=0)
print(y_output)

##### **Exercise 1 :**  Predict two steps of Fibonacci series using Keras TimeSeries Generator

#### Solution to exercise 1

<details>
    <summary> Click here for the answer </summary>
    
    from tensorflow import keras
    from keras.utils import timeseries_dataset_from_array
    from numpy import array
    from keras.models import Sequential
    from keras.layers import Dense, Input

    input_seq = array([0,1,1,2,3,5,8,13,21,34,55,89])
    output_seq = array( [input_seq[i:i+2] for i in range(3,input_seq.size-1)] )
    print('output_seq =\n', output_seq)

    seq_length = 3

    training_dataset = keras.utils.timeseries_dataset_from_array(
        data=input_seq[:],
        targets=output_seq[:],
        sequence_length=3,
        batch_size=2,
    )

    print('training_dataset = ')
    for inputs, targets in training_dataset:
        for i in range(inputs.shape[0]):
            print([int(x) for x in inputs[i]], [int(x) for x in targets[i]])

    # Model architecture
    model = Sequential()
    model.add(Input(shape = (seq_length,)))
    model.add(Dense(100, activation='relu'))
    model.add(Dense(2))

    # Model training
    model.compile(optimizer='adam', loss='mse')
    model.fit(training_dataset, epochs=200, verbose=0)

    # Model inference
    x_input = array([34, 55,89]).reshape((1, seq_length))
    y_output = model.predict(x_input, verbose=0)

    print('Two subsequent steps of Fibonnaci sequence after', x_input, 'are', y_output)
    
    
</details>

##### **Step 3**- Train sequences using Recurrent Neural Networks - Long Short Term Memory (LSTM)  and Gated Recurrent Unit (GRU)architectures.

The code below uses LSTM for training. The shape of training array is converted to (samples, timesteps, features).

Experiment: Change raw sequence to be a harmonic sequence and see if LSTM can forecast one step. Try updating the forecast to two steps by changing (X) and (y)

In [0]:
from keras.utils import timeseries_dataset_from_array
import tensorflow as tf
from numpy import array
from keras.models import Sequential
from keras.layers import LSTM
from keras.layers import Dense
from keras.layers import Input


#Harmonic Sequence for experimentation
harmonic_seq= [1, 0.5, 0.33, 0.25, 0.2, 0.17, 0.14, 0.125, 0.11]

# choose a number of time steps
n_steps = 3

# create training dataset
training_dataset = keras.utils.timeseries_dataset_from_array(
    data=harmonic_seq[:],
    targets=harmonic_seq[3:],
    sequence_length=3,
    batch_size=1,
)

# reshape from [samples, timesteps] into [samples, timesteps, features]
n_features = 1

# define model
model = Sequential()
model.add(Input(shape=(n_steps, n_features)))
model.add(LSTM(20, activation='relu'))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')
model.summary()

# fit model
model.fit(training_dataset, epochs=200, verbose=0)

# Input for experimentation on Harmonic sequence
x_input = array ([0.14, 0.125, 0.11])

x_input = x_input.reshape((1, n_steps, n_features))

y_output = model.predict(x_input, verbose=0)

print(f'predicted output from RNN-LSTM for Harmonic sequence {y_output}')

# Convert the Keras model to a TensorFlow Lite model
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS, # enable TensorFlow Lite built-in ops.
    tf.lite.OpsSet.SELECT_TF_OPS # enable select TensorFlow ops.
]
converter._experimental_lower_tensor_list_ops = False
tflite_model = converter.convert()

# Save the TensorFlow Lite model to a file
with open('lstm_harmonic.tflite', 'wb') as f:
    f.write(tflite_model)

print("LSTM model saved as lstm_harmonic.tflite")

##### Step 3(b)

The GRU model has lesser number of parameters and slightly better accuracy for predicting Harmonic sequence.

In [0]:
import tensorflow as tf
from numpy import array
from keras.models import Sequential
from keras.layers import LSTM
from keras.layers import GRU
from keras.layers import Dense, Input

# split a univariate sequence into samples
def split_sequence(sequence, n_steps):
	X, y = list(), list()
	for i in range(len(sequence)):
		# find the end of this pattern
		end_ix = i + n_steps
		# check if we are beyond the sequence
		if end_ix > len(sequence)-1:
			break
		# gather input and output parts of the pattern
		seq_x, seq_y = sequence[i:end_ix], sequence[end_ix]
		X.append(seq_x)
		y.append(seq_y)
	return array(X), array(y)


#Harmonic Sequence for experimentation
training_seq= [1, 0.5, 0.33, 0.25, 0.2, 0.17, 0.14, 0.125, 0.11]

# choose a number of time steps
n_steps = 3

# split into samples
X, y = split_sequence(training_seq, n_steps)

print ('X before reshape', X, X.shape)
print ('X.shape[0]', X.shape[0])
print ('X.shape[1]', X.shape[1])

# reshape from [samples, timesteps] into [samples, timesteps, features]
n_features = 1
X = X.reshape((X.shape[0], X.shape[1], n_features))

print ( X, X.shape)
print (y, y.shape)

# define model
model = Sequential()
model.add(Input(shape=(n_steps, n_features)))
model.add(GRU(20, activation='relu'))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')
model.summary()

# fit model
model.fit(X, y, epochs=200, verbose=0)


# Input for experimentation on Harmonic sequence
x_input = array ([0.14, 0.125, 0.11])

x_input = x_input.reshape((1, n_steps, n_features))

y_output = model.predict(x_input, verbose=0)

print(f'predicted output from RNN-GRU for Harmonic sequence {y_output}')

##### **Step 4** - Classify Time Series using CNN.

Use CNN to create signature of timeseries parameters

Create timeseries of 4 changing parameters.  This can represent motor speed, zone temperature and pressure of an Extruder.  
500 samples (m) are broken up to represent 5 zones of 100 samples each.  
A CNN network is trained to recognize the signature of each zone.

In [0]:
%matplotlib inline
from numpy import array
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from tensorflow.keras import layers
from numpy import hstack
import numpy as np
import matplotlib.pyplot as plt
from keras.layers import Input
from sklearn.preprocessing import StandardScaler

#traing samples, slope and offset
m = 500
train_slope_1  = 0.2
train_offset_1 = -1.0

train_slope_2 =  0.8
train_offset_2 = 1.0

train_slope_3 = -0.4
train_offset_3 = 0.5

train_slope_4 = 0.6
train_offset_4 = 0.5


def generate_dataset(n_points):
    x = np.linspace(0, 50, n_points).astype(np.float32)
    y1 = ((train_slope_1*x) + (2.0* np.sin(2*x)) + train_offset_1)
    y2 =  ((train_slope_2*x) + (4.0* np.sin(3*x)) + train_offset_2)
    y3 = ((train_slope_3*x) + (1.0* np.sin(4*x)) + train_offset_3)
    y4 = ((train_slope_4*x) + (6.0* np.cos(x)) + train_offset_4)
    return (x,y1,y2,y3,y4)

train_x,train_y1, train_y2, train_y3, train_y4= generate_dataset(m)

plt.rcParams["figure.figsize"] = (12, 8)

fig = plt.figure()
fig.suptitle('Training Time Series Data, Not Normalized', fontsize=20, fontweight='bold')

plt.plot(train_x, train_y1, 'k--');
plt.plot(train_x, train_y2, 'r--')
plt.plot(train_x, train_y3, 'g--')
plt.plot(train_x, train_y4, 'b--')
plt.show()

#Reshape training data
train_x = train_x.reshape(-1, 1)
train_y1 = train_y1.reshape(-1, 1)
train_y2 = train_y2.reshape(-1, 1)

train_y3 = train_y3.reshape(-1, 1)
train_y4 = train_y4.reshape(-1,1)

#Normalize training data using StandardScaler
scaler = StandardScaler()
train_y1 = scaler.fit_transform(train_y1)
train_y2 = scaler.fit_transform(train_y2)
train_y3 = scaler.fit_transform(train_y3)
train_y4 = scaler.fit_transform(train_y4)

#Assimilate training data as multivariate tensor
dataset = hstack((train_y1, train_y2, train_y3, train_y3))

#print dataset attributes
#print ('length of data is', len(dataset))
#print ('shape 0 of dataset is', dataset.shape)
#print ('shape 0 of dataset is', dataset.shape[0])
#print ('shape 1 of dataset is', dataset.shape[1])

dataset1 = dataset.reshape(-1,100,4)

X=dataset1

#Represent label (y) as one-hot encoded value
y = np.arange(5).reshape(-1,1)
y= tf.keras.utils.to_categorical(
    y, num_classes=None)


#Create Model
model = Sequential()
model.add(Input(shape=(100,4)))
model.add(layers.Conv1D(filters=64, kernel_size=5, activation='relu'))
model.add(layers.MaxPooling1D(pool_size=2))
model.add(layers.Conv1D(filters=128, kernel_size=3, activation='relu'))
model.add(layers.MaxPooling1D(pool_size=2))

model.add(layers.Flatten())
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dense(5, activation='softmax'))

model.compile(optimizer='adam', loss='categorical_crossentropy')
model.summary()

# Fit and save model
model.fit(X, y, epochs=200, verbose=0)

model.save("TS_model_1DCNN.keras")

##### **Step 5: Time Series Classification Inference:**

To determine which zone a particular set of timeseries parameters belongs to, we use model.predict to check the working of the model. Experiment by changing the zone (by tweaking **test_x** you can change the value of x and the corresponding zone).

In [0]:
#Inference on test data
from numpy import array
import tensorflow as tf
from tensorflow import keras
from numpy import hstack
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

#Load model
model = keras.models.load_model("TS_model_1DCNN.keras")

# Test slope and offset
# The slope and offset of the test data is not the same as training data
# Feel free to check the limits of the model by tweaking the test data slope, offset, frequency  and amplitude
test_m = 500
test_slope_1  = 0.5
test_offset_1 = -0.8

test_slope_2 =  0.3
test_offset_2 = 0.5

test_slope_3 = -0.9
test_offset_3 = 0.9

test_slope_4 = 0.1
test_offset_4 = 0.9


def generate_dataset(n_points):
    x = np.linspace(0, 50, n_points).astype(np.float32)
    y1 = ((test_slope_1*x) + (2.0 * np.sin(2*x)) + test_offset_1)
    y2 =  ((test_slope_2*x) + (4.0 * np.sin(3*x)) + test_offset_2)
    y3 = ((test_slope_3*x) + (1.0 * np.sin(4*x)) + test_offset_3)
    y4 = ((test_slope_4*x) + (6.0 * np.cos(x)) + test_offset_4)
    return (x,y1,y2,y3,y4)

test_x, test_y1, test_y2, test_y3, test_y4= generate_dataset(test_m)

plt.rcParams["figure.figsize"] = (12, 8)

fig = plt.figure()
fig.suptitle('Test Time Series Data, Not Normalized', fontsize=14)

plt.plot(train_x, test_y1, 'k--');
plt.plot(train_x, test_y2, 'r--')
plt.plot(train_x, test_y3, 'g--')
plt.plot(train_x, test_y4, 'b--')
plt.show()

#Reshape and Normalize test data
test_y1 = test_y1.reshape(-1, 1)
test_y2 = test_y2.reshape(-1, 1)

test_y3 = test_y3.reshape(-1, 1)
test_y4 = test_y4.reshape(-1,1)

scaler = StandardScaler()
test_y1 = scaler.fit_transform(test_y1)
test_y2 = scaler.fit_transform(test_y2)
test_y3 = scaler.fit_transform(test_y3)
test_y4 = scaler.fit_transform(test_y4)

#Join test features in one tensor

dataset_test = hstack((test_y1, test_y2, test_y3, test_y4))

plt.rcParams["figure.figsize"] = (12, 8)

#Split generated 500 point test data into 100 samples
test_1,test_2,test_3,test_4,test_5 = np.split(dataset_test,5,axis = 0)

#Check if the split is done correctly by checking length of any one section
print('length of section 4 is', len(test_4))

# Assign any section of the time series to find inference - test_1, test_2, test_3, test_4 or test_5
test = test_1.reshape((1,100,4))

x_input = array(test)

y_output = model.predict(x_input, verbose=0)
print('The softmax output with the highest value is the correct section', y_output)


##### **Review Questions**:

A.  Are Recurrent Neural Network (RNN) models based on LSTM layers advisable for use in constrained edge devices?

B. Are Convolutional Neural Network (CNN) layers suitable for edge devices?



### Answer

<details> 
    <summary> Click here to view our answer </summary>

    - LSTM layers are require considerable resources not available in many constrained edge devices. Some AI accelerators modules (such as Arm Ethos) do provide Unidirectional LSTM layer support.

    - CNN layers (both 1D and 2D) are supported by many edge devices and can be used for classification applications. The time-series classification model trained in the above cell is a good candidate for edge device.

</details>

#### **Exercise 2:**
- Convert 1D CNN time series Keras model to TFLite format and test is using inference APIs.
- TFLite does not support 1D CNN natively.  Use Netron to view the TFLite model to verify this.


#### Answer to Exercise 2

<details>
    <summary> Click here to view our solution </summary>

    import tensorflow as tf
    from tensorflow import keras

    # Load the Keras model
    model = keras.models.load_model("TS_model_1DCNN.keras")

    # Convert the Keras model to a TensorFlow Lite model
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS, # enable TensorFlow Lite built-in ops.
        tf.lite.OpsSet.SELECT_TF_OPS # enable select TensorFlow ops.
    ]
    converter._experimental_lower_tensor_list_ops = False
    tflite_model = converter.convert()

    # Save the TensorFlow Lite model to a file
    with open('TS_model_1DCNN.tflite', 'wb') as f:
        f.write(tflite_model)

    print("TS_model_1DCNN.keras converted and saved as TS_model_1DCNN.tflite")
    
</details>

#### Step 6: Run inference on Time Series TFLite Model

Install LiteRT runtime to execute TFLite model created.

In [0]:
!pip install -q ai-edge-litert

In [0]:
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

# Using TensorFlow's built-in Lite Interpreter
from ai_edge_litert.interpreter import Interpreter

# Load the TFLite model and allocate tensors
interpreter = Interpreter(model_path="TS_model_1DCNN.tflite")
interpreter.allocate_tensors()

# Get input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# --- Prepare test data (similar to cell jsUJpC4i6eZM) ---

# Test slope and offset - using the same as previous inference cell
test_m = 500
test_slope_1  = 0.5
test_offset_1 = -0.8

test_slope_2 =  0.3
test_offset_2 = 0.5

test_slope_3 = -0.9
test_offset_3 = 0.9

test_slope_4 = 0.1
test_offset_4 = 0.9

def generate_dataset(n_points):
    x = np.linspace(0, 50, n_points).astype(np.float32)
    y1 = ((test_slope_1*x) + (2.0 * np.sin(2*x)) + test_offset_1)
    y2 =  ((test_slope_2*x) + (4.0 * np.sin(3*x)) + test_offset_2)
    y3 = ((test_slope_3*x) + (1.0 * np.sin(4*x)) + test_offset_3)
    y4 = ((test_slope_4*x) + (6.0 * np.cos(x)) + test_offset_4)
    return (x,y1,y2,y3,y4)

_, test_y1, test_y2, test_y3, test_y4= generate_dataset(test_m)

# Reshape and Normalize test data
test_y1 = test_y1.reshape(-1, 1)
test_y2 = test_y2.reshape(-1, 1)
test_y3 = test_y3.reshape(-1, 1)
test_y4 = test_y4.reshape(-1,1)

scaler = StandardScaler()
test_y1 = scaler.fit_transform(test_y1)
test_y2 = scaler.fit_transform(test_y2)
test_y3 = scaler.fit_transform(test_y3)
test_y4 = scaler.fit_transform(test_y4)

# Join test features in one tensor
dataset_test = np.hstack((test_y1, test_y2, test_y3, test_y4))

# Split generated 500 point test data into 100 samples
# Using the first section (test_1) for inference as in the previous example
test_input_segment = np.split(dataset_test, 5, axis=0)[4]

# Reshape input to (1, 100, 4) and ensure float32 data type
input_data = test_input_segment.reshape(input_details[0]['shape']).astype(input_details[0]['dtype'])

# Set the tensor
interpreter.set_tensor(input_details[0]['index'], input_data)

# Invoke inference
interpreter.invoke()

# Get the output tensor
output_data = interpreter.get_tensor(output_details[0]['index'])

print(f"Predicted output from TFLite model: {output_data}")

#### **Exercise 3:**

Download (electromyogram) EMG dataset on various gestures. This dataset contains electrical activity of muscles when performing gestures. Formulate a plan to train a CNN based classifier for determining gestures. If you have time, train a classifier CNN model and create confusion matrix.

We start you off by obtaining the dataset and formatting the data as a Numpy array. 

In [0]:
%%script bash

wget -q https://edge-ai-doulos.s3.us-west-2.amazonaws.com/EMG_data_for_gestures-master.zip
unzip -q EMG_data_for_gestures-master.zip

In [0]:
import pandas as pd

# Read data from text file
df = pd.read_csv(r'EMG_data_for_gestures-master/01/1_raw_data_13-12_22.03.16.txt', delimiter = "\t")

# Data consists of 8 channels and last column is class of action
print (df.describe())
print (df.head(10))

# Convert Pandas to Numpy Array
my_array = df.to_numpy()

# Check conversion by printing shape of Numpy Array
print('Shape of Numpy array is', my_array.shape)

#### Solution to Exercise 3

<details>
    <summary> Click here to view our answer </summary>

    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from tensorflow.keras.utils import to_categorical
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Input
    import numpy as np
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    import matplotlib.pyplot as plt

    # Separate features (channels 1-8) and labels (class)
    X = my_array[:, 1:9]
    y = my_array[:, 9]

    # Normalize features
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Assuming each gesture is a single timestep for now.
    # If gestures have a sequence of readings, this needs adjustment.
    X = X.reshape(X.shape[0], 1, X.shape[1])

    # One-hot encode labels
    y = to_categorical(y)

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Define the 1D-CNN model
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))
    model.add(Conv1D(filters=64, kernel_size=1, activation='relu'))
    model.add(MaxPooling1D(pool_size=1))
    model.add(Flatten())
    model.add(Dense(100, activation='relu'))
    model.add(Dense(y_train.shape[1], activation='softmax')) # Output layer with number of classes

    # Compile the model
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    # Train the model
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)

    # Evaluate the model on the test data
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
    print(f'Test Accuracy: {accuracy*100:.2f}%')

    # Predict the classes for the test set
    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true_classes = np.argmax(y_test, axis=1)

    # Generate the confusion matrix
    cm = confusion_matrix(y_true_classes, y_pred_classes)

    # Display the confusion matrix
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap=plt.cm.Blues)
    plt.title('Confusion Matrix')
    plt.show()
    
</details>

Use this example time series [Colab notebook](https://colab.research.google.com/github/keras-team/keras-io/blob/master/examples/timeseries/ipynb/eeg_signal_classification.ipynb) from Keras that uses brainwave signals to classify different actions.